file taken from cv data from 5 class 5 cv 13 aug

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# CREATE HUMAN ANNOTATION FILES FOR CV1 LLM INTENT-PRESERVATION AUDIT
# Sample: 40 Chitchat + 40 Revise + 20 Inquire = 100 rows
# Annotators see only context + generated target turn.
# Source GT is hidden in master file only.
# ============================================================

import os
import json
import numpy as np
import pandas as pd

try:
    import openpyxl
except ImportError:
    !pip install openpyxl
    import openpyxl

from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

# -------------------------
# PATHS

DATA_PATH = "/content/drive/MyDrive/Human_annotation_intent/conversation_en.json"

SYNTHETIC_CV1_PATH = "/content/drive/MyDrive/Human_annotation_intent/cv1_synthetic_100_chitchat_100_revise_27_inquire.csv"

CV1_TRAIN_IDS_PATH = "/content/drive/MyDrive/Human_annotation_intent/cv1_train_ids.csv"

OUT_DIR = "/content/drive/MyDrive/Human_annotation_intent/annotation_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42

TARGET_SAMPLE = {
    "Chitchat": 40,
    "Revise": 40,
    "Inquire": 20,
}

VALID_LABELS = ["Chitchat", "Interpret", "Reveal", "Revise", "Inquire"]

# -------------------------
# LOAD FILES
# -------------------------
syn_df = pd.read_csv(SYNTHETIC_CV1_PATH)
train_ids_df = pd.read_csv(CV1_TRAIN_IDS_PATH)

syn_df["conversation_id"] = syn_df["conversation_id"].astype(str)
syn_df["turn_index"] = syn_df["turn_index"].astype(int)
syn_df["role"] = syn_df["role"].astype(str).str.lower()
syn_df["text"] = syn_df["text"].fillna("").astype(str)
syn_df["intent"] = syn_df["intent"].astype(str)
syn_df["source_user_text"] = syn_df["source_user_text"].fillna("").astype(str)
syn_df["source_system_text"] = syn_df["source_system_text"].fillna("").astype(str)

train_cids = set(train_ids_df["conversation_id"].astype(str))

print("Synthetic CV1 rows:", len(syn_df))
print("Synthetic distribution:")
print(syn_df["intent"].value_counts())

# -------------------------
# VERIFY SYNTHETIC ROWS ARE CV1 TRAIN-SOURCE ONLY
# -------------------------
syn_cids = set(syn_df["conversation_id"].astype(str))
outside_train = sorted(syn_cids - train_cids)

print("\nSynthetic source conversations:", len(syn_cids))
print("Synthetic conversations outside CV1 train:", len(outside_train))

assert len(outside_train) == 0, "Some synthetic rows do not belong to CV1 train IDs."

# -------------------------
# LOAD ORIGINAL ENGLISH CONVERSATIONS FOR FULL HISTORY
# -------------------------
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

all_rows = []
for conv in raw:
    cid = str(conv.get("conv_id"))
    for i, t in enumerate(conv.get("conversation", [])):
        all_rows.append({
            "conversation_id": cid,
            "turn_index": int(i),
            "role": str(t.get("role", "")).lower(),
            "text": t.get("content") or "",
            "intent": t.get("intent"),
        })

turns_full = (
    pd.DataFrame(all_rows)
    .sort_values(["conversation_id", "turn_index"])
    .reset_index(drop=True)
)

turns_full["conversation_id"] = turns_full["conversation_id"].astype(str)
turns_full["turn_index"] = turns_full["turn_index"].astype(int)
turns_full["role"] = turns_full["role"].astype(str).str.lower()
turns_full["text"] = turns_full["text"].fillna("").astype(str)

turns_by_conv = {
    cid: g.sort_values("turn_index").reset_index(drop=True)
    for cid, g in turns_full.groupby("conversation_id")
}

missing_context = sorted(syn_cids - set(turns_by_conv.keys()))
print("Synthetic conversations missing from original JSON:", len(missing_context))
assert len(missing_context) == 0, "Some source conversations are missing from conversation_en.json."

# -------------------------
# BUILD DIALOGUE HISTORY BEFORE TARGET TURN
# -------------------------
def build_dialogue_history_before_target(conversation_id, turn_index):
    conv_turns = turns_by_conv[str(conversation_id)]
    prev_turns = conv_turns[conv_turns["turn_index"] < int(turn_index)]

    history = []
    for _, p in prev_turns.iterrows():
        role = str(p["role"]).upper().strip()
        text = str(p["text"]).strip()
        if text:
            history.append(f"{role}: {text}")

    return "\n".join(history)

syn_df["dialogue_history_before_target"] = syn_df.apply(
    lambda r: build_dialogue_history_before_target(r["conversation_id"], r["turn_index"]),
    axis=1
)

# -------------------------
# STRATIFIED SAMPLE: 40 / 40 / 20
# -------------------------
sample_parts = []

for label, n in TARGET_SAMPLE.items():
    pool = syn_df[syn_df["intent"] == label].copy()
    print(f"{label}: available={len(pool)}, requested={n}")
    assert len(pool) >= n, f"Not enough {label} rows."

    sample_parts.append(
        pool.sample(n=n, random_state=RANDOM_STATE)
    )

audit_df = (
    pd.concat(sample_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

audit_df.insert(0, "audit_id", [f"CV1_AUDIT_{i+1:03d}" for i in range(len(audit_df))])

print("\nFinal audit sample distribution:")
print(audit_df["intent"].value_counts())
print("Total audit rows:", len(audit_df))

assert len(audit_df) == sum(TARGET_SAMPLE.values())

# -------------------------
# MASTER FILE: HAS HIDDEN GT / PROVENANCE FOR YOUR ANALYSIS
# -------------------------
master_df = audit_df[[
    "audit_id",
    "conversation_id",
    "turn_index",
    "augmentation_type",
    "intent",
    "dialogue_history_before_target",
    "source_system_text",
    "source_user_text",
    "text",
]].copy()

master_df = master_df.rename(columns={
    "conversation_id": "source_conversation_id",
    "turn_index": "source_turn_index",
    "intent": "source_gt_intent_hidden",
    "text": "generated_user_turn_to_label",
})

master_df["annotator_1_label"] = ""
master_df["annotator_2_label"] = ""
master_df["annotator_3_label"] = ""

master_path = os.path.join(OUT_DIR, "CV1_100_human_audit_MASTER_with_hidden_GT.xlsx")
master_df.to_excel(master_path, index=False)

# -------------------------
# ANNOTATOR-FACING FILES: NO SOURCE GT, NO MODEL PREDICTION
# -------------------------
annotator_df = master_df[[
    "audit_id",
    "dialogue_history_before_target",
    "generated_user_turn_to_label",
]].copy()

annotator_df["annotator_label"] = ""
instructions = pd.DataFrame({
    "Instruction": [
        "Task: Label the intent of the GENERATED user turn only, using the dialogue history before it.",
        "Do not infer the original/source label. The source label and model prediction are hidden intentionally.",
        "Choose exactly one label: Chitchat, Interpret, Reveal, Revise, Inquire."
    ]
})

definitions = pd.DataFrame({
    "Intent": VALID_LABELS,
    "Definition": [
        "Chitchat: Social or conversational utterances not directly advancing the product-search task, such as greetings, thanks, goodbye, or small talk.",

        "Interpret: The user expresses, clarifies, or confirms their current product-search need in response to the system, without introducing a new constraint or changing a previous preference.",

        "Reveal: The user introduces a new product preference, requirement, constraint, or selection criterion that was not previously stated in the dialogue.",

        "Revise: The user changes, corrects, replaces, or contradicts a previously stated preference, requirement, constraint, or selection.",

        "Inquire: The user asks a question or requests information, clarification, comparison, availability, explanation, or product details."
    ]
})

for annotator_id in [1, 2, 3]:
    out_path = os.path.join(OUT_DIR, f"CV1_100_human_audit_ANNOTATOR_{annotator_id}.xlsx")

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        instructions.to_excel(writer, sheet_name="Instructions", index=False)
        definitions.to_excel(writer, sheet_name="Intent_Definitions", index=False)
        annotator_df.to_excel(writer, sheet_name="Annotation", index=False)

    wb = load_workbook(out_path)
    ws = wb["Annotation"]

    # Freeze header and wrap text
    ws.freeze_panes = "A2"

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.fill = PatternFill("solid", fgColor="D9EAF7")
        cell.alignment = Alignment(wrap_text=True, vertical="top")

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")

    # Column widths
    widths = {
        "A": 16,  # audit_id
        "B": 70,  # history
        "C": 55,  # generated turn
        "D": 18,  # label
    }

    for col, width in widths.items():
        ws.column_dimensions[col].width = width

    # Data validation dropdowns
    label_dv = DataValidation(
        type="list",
        formula1='"Chitchat,Interpret,Reveal,Revise,Inquire"',
        allow_blank=False
    )
    ws.add_data_validation(label_dv)

    label_dv.add(f"D2:D{len(annotator_df)+1}")

    wb.save(out_path)
    print("Saved annotator file:", out_path)

print("\nSaved master file:", master_path)
print("All files saved to:", OUT_DIR)

# -------------------------
# FINAL CHECKS
# -------------------------
print("\nCHECK: annotators do NOT see source GT label.")
print("Annotator-facing columns:")
print(list(annotator_df.columns))

print("\nMaster hidden columns:")
print(list(master_df.columns))

Synthetic CV1 rows: 227
Synthetic distribution:
intent
Chitchat    100
Revise      100
Inquire      27
Name: count, dtype: int64

Synthetic source conversations: 76
Synthetic conversations outside CV1 train: 0
Synthetic conversations missing from original JSON: 0
Chitchat: available=100, requested=40
Revise: available=100, requested=40
Inquire: available=27, requested=20

Final audit sample distribution:
intent
Revise      40
Chitchat    40
Inquire     20
Name: count, dtype: int64
Total audit rows: 100
Saved annotator file: /content/drive/MyDrive/Human_annotation_intent/annotation_outputs/CV1_100_human_audit_ANNOTATOR_1.xlsx
Saved annotator file: /content/drive/MyDrive/Human_annotation_intent/annotation_outputs/CV1_100_human_audit_ANNOTATOR_2.xlsx
Saved annotator file: /content/drive/MyDrive/Human_annotation_intent/annotation_outputs/CV1_100_human_audit_ANNOTATOR_3.xlsx

Saved master file: /content/drive/MyDrive/Human_annotation_intent/annotation_outputs/CV1_100_human_audit_MASTER_with